<a href="https://colab.research.google.com/github/ankit-rathi/Quantvesting_v2/blob/main/myStocks_Portfolio_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
'''
!pip install pandas==2.2.2 -qq
!pip install pyxirr -qq
!pip install ta -qq
!pip install yfinance==0.2.59 -qq
'''

'\n!pip install pandas==2.2.2 -qq\n!pip install pyxirr -qq\n!pip install ta -qq\n!pip install yfinance==0.2.59 -qq\n'

In [2]:
# import libraries

import numpy as np
import pandas as pd
import datetime
import warnings
warnings.filterwarnings('ignore')
import yfinance as yfin
import ta
import matplotlib.pyplot as plt
import requests

In [3]:
# notebook start time
import time
import datetime
import pytz

start_time = time.time()

# function to print date timestamp
def print_date_time():
  tz_NY = pytz.timezone('Asia/Kolkata')
  datetime_NY = datetime.datetime.now(tz_NY)
  print("Run date time (IST):", datetime_NY.strftime("%Y-%m-%d %H:%M:%S"))

In [4]:
# function to get booked and reserved amount
def get_amt():
  # fetch the JSON data from the URL
  url = "https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myPortfolioAmts.json"
  response = requests.get(url)
  pf_amts = response.json()  # parse the JSON data

  # extract values from the JSON
  py_booked_amt_dm = pf_amts["py_booked_amt_dm"]
  py_booked_amt_sv = pf_amts["py_booked_amt_sv"]
  cy_booked_amt_dm = pf_amts["cy_booked_amt_dm"]
  cy_booked_amt_sv = pf_amts["cy_booked_amt_sv"]
  reserve_amt_dm = pf_amts["reserve_amt_dm"]
  reserve_amt_sv = pf_amts["reserve_amt_sv"]

  # perform calculations
  py_booked_amt = py_booked_amt_dm + py_booked_amt_sv
  cy_booked_amt = cy_booked_amt_dm + cy_booked_amt_sv
  reserve_amt = reserve_amt_dm + reserve_amt_sv
  total_booked_amt = py_booked_amt + cy_booked_amt

  return total_booked_amt, reserve_amt, cy_booked_amt, py_booked_amt

gen_output = 0

In [5]:
# function to format the amount
def format_amt(number):
    abs_number = abs(number)

    if abs_number >= 1_00_00_000:  # Crores
        formatted_amt = f"{number / 1_00_00_000:.2f} C"
    elif abs_number >= 1_00_000:  # Lakhs
        formatted_amt = f"{number / 1_00_000:.2f} L"
    elif abs_number >= 1_000:  # Thousands
        formatted_amt = f"{number / 1_000:.2f} K"
    else:
        formatted_amt = f"{number:.2f}"

    return formatted_amt

# function to set start and end date
def get_start_end_date():
  start_date = (datetime.date.today() + datetime.timedelta(days=-365)).strftime('%Y-%m-%d')
  end_date = (datetime.date.today() + datetime.timedelta(days=1)).strftime('%Y-%m-%d')
  return start_date, end_date

# function to fetch my portfolio csv
def get_mypfs_df():
  mypfs_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myPortfolioStocks.csv')
  return mypfs_df

# function to fetch my prospects csv
def get_mypps_df():
  mypps_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myProspectsScrips.csv')
  return mypps_df

# function to fetch screener data
def get_myscreen_df():
  myscreen_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myScreenerDB.csv')
  return myscreen_df

# function to fetch momentum data
def get_myinvmt_df():
  myinvmt_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myInvestments.csv')
  return myinvmt_df

# function to get the stock ids
def get_stock_ids(df_pf):
  stock_n100 = df_pf['Symbol'].unique()

  exclude = ['CADILAHC','MMTC', 'MASFIN']

  stock_ids = df_pf[~df_pf['Symbol'].isin(exclude) ]['Symbol'].unique()

  #mypf = mypf[mypf.Forecast.notnull()]
  #stock_ids = mypf['Symbol'].unique()

  #stock_ids.sort()
  return stock_ids

# get features from screener data
def get_screener_features():
  myscreen_df = get_myscreen_df()
  cols = ['Symbol', 'EPS', 'MedPE', 'ROCE%', 'ROE%', 'CapType']
  return myscreen_df[cols]

# get relative strength
def get_relative_strength(stock_list):

    # Dictionary to store stock tickers and their corresponding percentage price change
    stock_changes = {}

    # Loop through each stock and fetch its price data
    for stock in stock_list:
        try:
            # Download the stock data for the given date range
            stock_data = yfin.Ticker(stock + '.NS').history(period='1mo', interval='1d')[map(str.title, ['open', 'close', 'low', 'high', 'volume'])]

            # Calculate the percentage change for the stock
            if len(stock_data) > 0:
                start_price = stock_data['Close'].iloc[0]
                end_price = stock_data['Close'].iloc[-1]
                percent_change = round(((end_price - start_price) / start_price) * 100, 2)
                stock_changes[stock] = percent_change
            else:
                stock_changes[stock] = np.nan  # If no data is available, set to NaN

        except Exception as e:
            print(f"Error fetching data for {stock}: {e}")
            stock_changes[stock] = np.nan

    # Create a DataFrame with stock tickers and their percentage changes
    df = pd.DataFrame(list(stock_changes.items()), columns=['Symbol', 'Percent_Change'])

    # Drop any stocks with missing data (NaN values)
    df = df.dropna()

    # Calculate the percentile rank based on percentage change
    df['RSP'] = round(df['Percent_Change'].rank(pct=True) * 100, 2)
    cols = ['Symbol', 'RSP']
    df = df[cols]
    # Sort by percentile rank (optional)
    df = df.sort_values(by='RSP', ascending=False).reset_index(drop=True)

    return df


In [6]:
# function to get stock technicals
def stock_prec_dev(stock_symbol):
    #stock_symbol = 'ULTRACEMCO.NS'
    short_window = 20
    mid_window = 50
    long_window = 200
    moving_avg = 'SMA'

    start = datetime.datetime(*map(int, start_date.split('-')))
    end = datetime.datetime(*map(int, end_date.split('-')))

    stock_df = yfin.Ticker(stock_symbol + '.NS').history(period='max', interval='1d')[map(str.title, ['open', 'close', 'low', 'high', 'volume'])]

    stock_df['Max'] = round(max(stock_df['Close']),2)
    stock_df = stock_df[(stock_df.index <= end_date) & (stock_df.index >= start_date)]
    stock_df['200_SMA'] = round(stock_df['Close'].rolling(window = 200, min_periods = 1).mean(),0)
    stock_df['Dev%_200'] = round((stock_df['Close'] - stock_df['200_SMA'])*100/stock_df['200_SMA'],2)
    stock_df.dropna(axis = 0, inplace = True) # remove any null rows

    stock_df['20_SMA'] = stock_df['Close'].rolling(window=20).mean()
    stock_df['50_SMA'] = stock_df['Close'].rolling(window=50).mean()
    stock_df['Symbol'] = stock_symbol

    stock_df['Close'] = round(stock_df['Close'],2)
    stock_df['Min'] = round(min(stock_df['Close']),2)
    stock_df['RSI_14'] = ta.momentum.RSIIndicator(close=stock_df['Close'], window=14).rsi()
    stock_df['RSI_14'] = round(stock_df['RSI_14'],0)
    stock_df['Prev_Close'] = stock_df['Close'].shift(1)
    stock_df.drop(['Open', 'Low', 'High', 'Volume'], axis=1, inplace=True)
    stock_df = stock_df.tail(1)

    max_SMA = max(stock_df['20_SMA'].item(), stock_df['50_SMA'].item(), stock_df['200_SMA'].item())
    min_SMA = min(stock_df['20_SMA'].item(), stock_df['50_SMA'].item(), stock_df['200_SMA'].item())
    ABS_Spread = max_SMA - min_SMA
    stock_df['Spread%'] = round((ABS_Spread / stock_df['200_SMA'].item()) * 100,2)

    return stock_df

# function to compute stock attributes
def get_common_features(stock_ids, df_mypf):

  df_prec_dev = pd.DataFrame()
  df_tmp = get_screener_features()
  df_rs = get_relative_strength(stock_ids)

  for stock_id in stock_ids:
      tmp = stock_prec_dev(stock_id)
      tmp = tmp.reset_index()
      df_prec_dev = pd.concat([df_prec_dev, tmp], ignore_index = True)
  df_prec_dev = pd.merge(df_prec_dev, df_mypf, on= 'Symbol')
  df_prec_dev = pd.merge(df_prec_dev, df_tmp, on= 'Symbol', how='left')
  df_prec_dev = pd.merge(df_prec_dev, df_rs, on= 'Symbol', how='left')
  #print(df_prec_dev.columns)
  df_prec_dev['Curr_PE'] = round(df_prec_dev['Close']/df_prec_dev['EPS'],1)
  df_prec_dev['Dev%_PE'] = round((df_prec_dev['Curr_PE'] - df_prec_dev['MedPE'])*100/df_prec_dev['MedPE'],2)
  df_prec_dev['Conviction'] = df_prec_dev['Conviction'] + '-' + df_prec_dev['CapType']
  return df_prec_dev

# function to arrange stock features
def arrange_features(df_stocks, common_cols, diff_cols):
  df_stocks_common = df_stocks[common_cols].drop_duplicates()
  df_stocks_diff = df_stocks[diff_cols]
  df_stocks_diff['Investment'] = df_stocks_diff['AvgCost'] * df_stocks_diff['Shares']
  df_stocks_diff = df_stocks_diff.groupby(['Symbol'])[['Shares', 'Investment']].aggregate(['sum']).reset_index()
  df_stocks_diff.columns = ['Symbol', 'Shares', 'Investment']
  df_stocks_diff['AvgCost'] = round(df_stocks_diff['Investment']/df_stocks_diff['Shares'],2)
  df_stocks = pd.merge(df_stocks_diff, df_stocks_common, on='Symbol')
  return df_stocks

# plot fact distribution across dimension
def plot_pie_chart(df, dimension, fact):
  # grouping the data by category and calculating the sum of fact for each type
  grouped_data = df.groupby(dimension)[fact].sum()

  # sorting the grouped data in descending order
  grouped_data = grouped_data.sort_values(ascending=False)

  # creating a pie chart
  grouped_data.plot.pie(autopct='%1.1f%%', startangle=90, figsize=(6, 6))

  # adding a title and displaying the plot
  plt.title(f'{dimension} {fact} Distribution')
  plt.ylabel('')  # To hide the y-label
  plt.show()

In [7]:
# function to get portfolio features

def get_portfolio_features(df_common_features):

  print_date_time()
  print('=================== PORTFOLIO QUALITY ====================')

  #df_common_features["Target"] = df_common_features["Target"].fillna(df_common_features["Max"])
  df_common_features['NTT'] = np.where(df_common_features['Strategy']== 'NTT', df_common_features["Target"], df_common_features['Max'])
  df_common_features['LTT'] = np.where(df_common_features['Strategy']== 'BTT', df_common_features["Target"], df_common_features['Max'])
  df_common_features['BOL'] = df_common_features['Min']

  tmp_df = df_common_features[df_common_features['Symbol'].isin(mypf_df[mypf_df['InPortfolio'] != 'NA'].Symbol.values)]
  print('qualified stocks: '+str(len(tmp_df['Symbol'].unique())))
  tmp_df1 = tmp_df[tmp_df['LatestQtr'] == 1]
  print('with latest results: '+str(len(tmp_df1['Symbol'].unique())))
  tmp_df1 = tmp_df1[tmp_df1['StarStock'] == 1]
  print('still star stocks: '+str(len(tmp_df1['Symbol'].unique())))
  tmp_df['Investment'] = tmp_df['AvgCost'] * tmp_df['Shares']
  tmp_df['Current'] = round(tmp_df['Close'] * tmp_df['Shares'],0)
  tmp_df['Previous'] = tmp_df['Prev_Close'] * tmp_df['Shares']
  tmp_df['EstimatedST'] = tmp_df['NTT'] * tmp_df['Shares']
  tmp_df['EstimatedLT'] = tmp_df['LTT'] * tmp_df['Shares']
  tmp_df['Current P/L'] = round((tmp_df['Current'] - tmp_df['Investment']),0)
  tmp_df['Today P/L%'] = round((tmp_df['Current'] - tmp_df['Previous'])*100/tmp_df['Previous'],2)
  tmp_df['Current P/L%'] = round((tmp_df['Current'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['EstimatedST P/L%'] = round((tmp_df['EstimatedST'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['EstimatedLT P/L%'] = round((tmp_df['EstimatedLT'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['NTT%'] = round((tmp_df['NTT'] - tmp_df['Close'])*100/tmp_df['Close'],2)
  tmp_df['LTT%'] = round((tmp_df['LTT'] - tmp_df['Close'])*100/tmp_df['Close'],2)
  tmp_df['Gained%'] = round((tmp_df['Close'] - tmp_df['BOL'])*100/tmp_df['BOL'],2)
  investment = round(sum(tmp_df['AvgCost']*tmp_df['Shares']),0)
  current = round(sum(tmp_df['Close']*tmp_df['Shares']),0)
  tmp_df['InitAlloc%'] = round(tmp_df['Investment']*100/investment,2)
  tmp_df['CurrAlloc%'] = round(tmp_df['Current']*100/current,2)
  tmp_df['FTT'] = tmp_df['LTT']
  tmp_df.loc[tmp_df['Strategy'] == 'NTT', 'FTT'] = tmp_df['NTT']
  tmp_df['FTT%'] = tmp_df['LTT%']
  tmp_df.loc[tmp_df['Strategy'] == 'NTT', 'FTT%'] = tmp_df['NTT%']
  tmp_df['FTT Amt'] = round(tmp_df['FTT%'] * tmp_df['Current']/100,0)
  tmp_df['OTT%'] = round((tmp_df['FTT'] - tmp_df['AvgCost'])*100/tmp_df['AvgCost'],2)
  tmp_df['RRR Ind'] = round(tmp_df['Current P/L']/tmp_df['FTT Amt'],2)
  tmp_df['Risk Ind'] = round(tmp_df['Current P/L%']*tmp_df['CurrAlloc%'],0)

  return tmp_df

# function to print portfolio features
def print_portfolio_stats(df_portfolio_features, myinvmt_df):
  from pyxirr import xirr

  total_booked_amt, reserve_amt, cy_booked_amt, py_booked_amt = get_amt()

  dates = myinvmt_df['Date'].values
  dates = np.append(dates, datetime.date.today().strftime('%d-%b-%y'))
  investment = myinvmt_df['Investment'].values
  dates= pd.to_datetime(dates)

  current = round(sum(df_portfolio_features['Close']*df_portfolio_features['Shares']),0) + reserve_amt
  investment_xirr = np.append(investment, current)
  cagr = round(xirr(pd.DataFrame({"dates": dates, "amounts": investment_xirr}))*100,2)

  investment = -sum(investment)
  invested = round(sum(df_portfolio_features['AvgCost']*df_portfolio_features['Shares']),0) + reserve_amt
  previous = round(sum(df_portfolio_features['Prev_Close']*df_portfolio_features['Shares']),0) + reserve_amt
  cy_invested = investment + py_booked_amt

  today_pnl_amount = current-previous
  today_pnl_percentage = round((current-previous)*100/previous,2)

  curr_pnl_amount = current-invested
  curr_pnl_percentage = round((curr_pnl_amount)*100/(cy_invested),2)

  cy_pnl_amount = cy_booked_amt + curr_pnl_amount
  cy_pnl_percentage = round((cy_pnl_amount)*100/cy_invested,2)

  overall_pnl_amount = total_booked_amt + curr_pnl_amount
  overall_pnl_percentage = round((overall_pnl_amount)*100/investment,2)

  estimate_st = round(sum(df_portfolio_features['FTT']*df_portfolio_features['Shares']),0)  + reserve_amt
  est_st_pnl_amount = estimate_st-current
  est_st_pnl_percentage = round((est_st_pnl_amount)*100/current,2)

  estimate_lt = round(sum(df_portfolio_features['LTT']*df_portfolio_features['Shares']),0)  + reserve_amt
  est_lt_pnl_amount = estimate_lt-current
  est_lt_pnl_percentage = round((est_lt_pnl_amount)*100/current,2)

  total_profit = round(sum(df_portfolio_features[df_portfolio_features['Current P/L%'] > 0]['Current']) - sum(df_portfolio_features[df_portfolio_features['Current P/L%'] > 0]['Investment']),0)
  total_loss = round(sum(df_portfolio_features[df_portfolio_features['Current P/L%'] < 0]['Current']) - sum(df_portfolio_features[df_portfolio_features['Current P/L%'] < 0]['Investment']),0)

  cy_booked_percentage = round((cy_booked_amt)*100/current,2)
  py_booked_percentage = round((py_booked_amt)*100/investment,2)
  total_booked_percentage = round((total_booked_amt)*100/investment,2)

  print('=================== PORTFOLIO SNAPSHOT ===================')
  print('Deployed Value: ', format_amt(investment))
  print('Current Value:  ', format_amt(current))
  print('CAGR/XIRR %:    ', '{}%'.format(cagr))
  print('Est FTT Value:  ', format_amt(estimate_st))
  print('Future Upside:  ', '{} ({}%)'.format(
      format_amt(est_st_pnl_amount),
      est_st_pnl_percentage
  ))

  print('=================== PERFORMANCE ==========================')
  print('Current PnL:             {} ({}%)'.format(
      format_amt(curr_pnl_amount), curr_pnl_percentage
  ))
  print('Total Booked PnL:        {} ({}%)'.format(
      format_amt(total_booked_amt), total_booked_percentage
  ))
  print('Booked + Current PnL:    {} ({}%)'.format(
      format_amt(overall_pnl_amount), overall_pnl_percentage
  ))
  print('CY Booked PnL:           {} ({}%)'.format(
      format_amt(cy_booked_amt), cy_booked_percentage
  ))
  print('Prev Year Booked PnL:    {} ({}%)'.format(
      format_amt(py_booked_amt), py_booked_percentage
  ))

  print('=================== TODAY / OPERATIONS ===================')
  print('Today PnL:               {} ({}%)'.format(
      format_amt(today_pnl_amount), today_pnl_percentage
  ))
  print('CY Investment:           ', format_amt(cy_invested))
  print('Reserve:                 ', format_amt(reserve_amt))

  print('=================== P&L DETAIL ============================')
  print('Total Profit:            ', format_amt(total_profit))
  print('Total Loss:              ', format_amt(total_loss))
  print('============================================================')

In [8]:
# get start and end date
start_date, end_date = get_start_end_date()

# get portfolio and prospects data
mypfs_df = get_mypfs_df()
mypps_df = get_mypps_df()
myinvmt_df = get_myinvmt_df()

# merge above datasets
mypf_df = pd.merge(mypfs_df, mypps_df, on="Symbol")

# seggregate dm and sv portfolio
dm_pf = mypf_df[mypf_df['InPortfolio'] == 'DM']
sv_pf = mypf_df[mypf_df['InPortfolio'] == 'SV']

# build portfolio stock dataframe
dm_stocks = get_stock_ids(dm_pf)
sv_stocks = get_stock_ids(sv_pf)
df_stocks = pd.concat([dm_pf,sv_pf], ignore_index = True)

# arrange common and diff stock features
common_cols = ['Symbol', 'Target', 'Criteria', 'Strategy', 'CumlRnk', 'LatestQtr', 'StarStock', 'Conviction', 'Category', 'InFolio', 'XIRR', 'MBQ']
diff_cols = ['Symbol', 'AvgCost', 'Shares']
df_stocks = arrange_features(df_stocks, common_cols, diff_cols)

# get common features
stock_ids = df_stocks['Symbol'].values
df_common_features = get_common_features(stock_ids, df_stocks)

df_common_features.reset_index(drop=True, inplace=True)
df_common_features.drop(['Date'], axis=1, inplace=True)
# get and print portfolio features
df_portfolio_features = get_portfolio_features(df_common_features)

print_portfolio_stats(df_portfolio_features, myinvmt_df)

df = df_portfolio_features
#plot_pie_chart(df, 'CapType', 'Current')
list_ox40n = list(df[df['MBQ'].str.contains('OX40N', na=False)]['Symbol'].values)

Run date time (IST): 2026-11-12 14:48:55
=================== PORTFOLIO QUALITY ====================
qualified stocks: 84
with latest results: 84
still star stocks: 44


=================== PORTFOLIO SNAPSHOT ===================
Deployed Value:  1.49 C
Current Value:   1.55 C
CAGR/XIRR %:     1.61%
Est FTT Value:   2.74 C
Future Upside:   1.19 C (76.62%)
=================== PERFORMANCE ==========================
Current PnL:             -40.30 L (-21.0%)
Total Booked PnL:        49.23 L (33.02%)
Booked + Current PnL:    8.93 L (5.99%)
CY Booked PnL:           6.36 L (4.1%)
Prev Year Booked PnL:    42.87 L (28.76%)
=================== TODAY / OPERATIONS ===================
Today PnL:               -16.73 K (-0.11%)
CY Investment:            1.92 C
Reserve:                  2.04 K
=================== P&L DETAIL ============================
Total Profit:             2.39 L
Total Loss:               -42.68 L


In [9]:
# top 5 near their targets
cols = ['Symbol', 'Today P/L%', 'Current P/L%', 'FTT%', 'OTT%', 'FTT Amt', 'Current P/L', 'Current', 'FTT', 'Dev%_PE', 'RSI_14','Conviction', 'Spread%',  'CumlRnk', 'RRR Ind', 'CurrAlloc%', 'Gained%', 'Criteria', 'Strategy', 'Category']
df_tmp = df_portfolio_features[~df_portfolio_features['Symbol'].isin(['ENRIN','BLUSPRING','DIGITIDE'])]
df_tmp = df_tmp.sort_values(by = 'FTT Amt', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
75,TTKPRESTIG,-1.16,-24.37,32.35,0.09,24656.0,-24561.0,76216.0,770.00,-8.02,62.0,M-SC,1.84,250.0,-1.00,0.49,37.18,OX40N,NTT,DURABLES
12,BOSCH-HCIL,-0.50,-22.56,29.12,-0.01,28821.0,-28831.0,98974.0,2282.00,775.75,52.0,M-SC,19.74,231.0,-1.00,0.64,68.47,OX40N,BTT,AC
50,M&M,-2.17,5.83,11.01,17.48,35082.0,17566.0,318641.0,3762.88,-7.77,74.0,X-LC,5.67,25.0,0.50,2.06,23.21,X40N,ATH,AUTO
65,SIS,-0.46,-10.45,35.42,21.26,35216.0,-11608.0,99424.0,528.00,91.06,49.0,H-SC,7.07,168.0,-0.33,0.64,47.83,OX40N,NTT,MISC
76,UNITDSPR,0.55,4.14,15.96,20.77,38918.0,9700.0,243844.0,1644.00,-11.27,58.0,X-MC,2.41,64.0,0.25,1.57,17.21,X40N,NTT,BREWERIES


In [10]:
# top 5 to accumulate
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC'])]
df_tmp = df_tmp[(df_tmp['CumlRnk'] < 40) & (df_tmp['FTT%'] > 20)].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
38,HYUNDAI,-2.26,7.79,32.01,42.29,97842.0,22089.0,305660.0,2782.73,12.96,60.0,X-LC,3.38,29.0,0.23,1.97,25.82,X40N,ATH,AUTO
51,MARUTI,-1.40,-4.33,31.69,25.99,102992.0,-14710.0,325000.0,17120.06,-10.97,70.0,X-LC,10.73,33.0,-0.14,2.10,16.38,XR,ATH,AUTO
47,JIOFIN,0.11,-18.82,67.63,36.09,220456.0,-75571.0,325974.0,387.00,-37.82,65.0,X-LC,9.48,34.0,-0.34,2.10,12.89,XY24,BTT,FINANCE
4,ASIANPAINT,-0.71,-7.64,35.16,24.83,115742.0,-27231.0,329186.0,3422.40,-19.51,64.0,X-LC,4.58,30.0,-0.24,2.12,20.69,X40,ATH,PAINTS
53,NESTLEIND,-1.88,6.05,21.85,29.22,72349.0,18878.0,331116.0,1540.00,-19.91,49.0,X-LC,8.80,20.0,0.26,2.14,8.83,XY25,ATH,FMCG


In [11]:
# top 5 today
df_tmp = df_portfolio_features.sort_values(by = 'Today P/L%', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
67,SURYODAY,6.03,-9.19,35.48,23.03,57697.0,-16452.0,162619.0,216.00,-25.86,68.0,H-SC,5.82,166.0,-0.29,1.05,38.03,XR,NTT,BANKS
69,TANLA,5.83,-16.93,159.04,115.17,357697.0,-45851.0,224910.0,1903.93,NaN,83.0,NaN,18.75,147.0,-0.13,1.45,98.82,AR,ATH,IT
41,INDIAMART,4.92,-30.13,187.57,100.92,161635.0,-37163.0,86173.0,4675.60,NaN,56.0,NaN,16.62,140.0,-0.23,0.56,9.33,AR,ATH,MISC
35,HDFCLIFE,3.91,3.43,36.15,40.83,129046.0,11842.0,356974.0,811.41,-24.85,74.0,X-LC,9.02,31.0,0.09,2.30,16.03,X40,ATH,INSURANCE
79,VALIANTORG,3.56,-22.61,125.03,74.15,158308.0,-36989.0,126616.0,838.00,NaN,42.0,NaN,37.83,145.0,-0.23,0.82,88.42,XR,NTT,CHEMICALS


In [12]:
# bottom 5 today
df_tmp = df_portfolio_features.sort_values(by = 'Today P/L%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
54,PAGEIND,-9.99,-14.15,54.75,32.85,126147.0,-37974.0,230405.0,50936.07,-36.12,35.0,X-MC,2.81,55.0,-0.30,1.49,10.02,X40,ATH,APPARELS
27,GLAXO,-2.56,6.70,27.17,35.69,57772.0,13348.0,212632.0,3380.16,-19.92,55.0,X-MC,8.18,56.0,0.23,1.37,26.37,X40,ATH,PHARMA
58,RECLTD,-2.33,-17.91,47.07,20.74,78507.0,-36382.0,166788.0,446.00,-9.91,54.0,M-MC,10.63,191.0,-0.46,1.08,7.11,XY25,NTT,FINANCE
38,HYUNDAI,-2.26,7.79,32.01,42.29,97842.0,22089.0,305660.0,2782.73,12.96,60.0,X-LC,3.38,29.0,0.23,1.97,25.82,X40N,ATH,AUTO
48,KANSAINER,-2.26,-32.73,68.68,13.47,124594.0,-88254.0,181413.0,340.00,-29.46,61.0,H-SC,6.05,162.0,-0.71,1.17,28.23,XY24,NTT,PAINTS


In [13]:
# top 5 to exit based on CumlRnk
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['Current P/L%'] > -1) & (df_tmp['Current P/L%'] < 1)].sort_values(by = 'CumlRnk', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category


In [14]:
# OX40N stocks to exit
df_tmp = df_portfolio_features[(df_portfolio_features['Symbol'].isin(list_ox40n))].sort_values(by = 'Current P/L%', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,2.77,59.50,29.43,106.44,69304.0,87845.0,235489.0,1629.94,16.37,52.0,M-SC,23.81,220.0,1.27,1.52,78.43,OX40N,ATH,CABLES
65,SIS,-0.46,-10.45,35.42,21.26,35216.0,-11608.0,99424.0,528.00,91.06,49.0,H-SC,7.07,168.0,-0.33,0.64,47.83,OX40N,NTT,MISC
12,BOSCH-HCIL,-0.50,-22.56,29.12,-0.01,28821.0,-28831.0,98974.0,2282.00,775.75,52.0,M-SC,19.74,231.0,-1.00,0.64,68.47,OX40N,BTT,AC
75,TTKPRESTIG,-1.16,-24.37,32.35,0.09,24656.0,-24561.0,76216.0,770.00,-8.02,62.0,M-SC,1.84,250.0,-1.00,0.49,37.18,OX40N,NTT,DURABLES
42,INDIGOPNTS,1.33,-30.32,43.55,0.02,52968.0,-52934.0,121625.0,1408.00,-33.47,49.0,M-SC,7.60,222.0,-1.00,0.79,38.58,OX40N,NTT,PAINTS


In [15]:
# top 5 to book for rotation
df_tmp = df_portfolio_features[(df_portfolio_features['Current P/L%'] > 20) ].sort_values(by = 'CumlRnk', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,2.77,59.5,29.43,106.44,69304.0,87845.0,235489.0,1629.94,16.37,52.0,M-SC,23.81,220.0,1.27,1.52,78.43,OX40N,ATH,CABLES


In [16]:
# top 5 to monitor for exit
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['Current P/L%'] > 0) ].sort_values(by = 'Dev%_PE', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,2.77,59.50,29.43,106.44,69304.0,87845.0,235489.0,1629.94,16.37,52.0,M-SC,23.81,220.0,1.27,1.52,78.43,OX40N,ATH,CABLES
66,STARHEALTH,2.72,10.88,32.81,47.26,102273.0,30589.0,311712.0,761.00,-2.33,60.0,H-SC,4.16,173.0,0.30,2.01,34.78,XY24,NTT,INSURANCE


In [17]:
# top 5 to book for rotation from weak conviction
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[~df_tmp['Criteria'].isin(['XY25','XY24','X40', 'X40N'])].sort_values(by = 'RRR Ind', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,2.77,59.50,29.43,106.44,69304.0,87845.0,235489.0,1629.94,16.37,52.0,M-SC,23.81,220.0,1.27,1.52,78.43,OX40N,ATH,CABLES
83,WIPRO,-1.30,-10.20,114.93,93.01,258273.0,-25518.0,224722.0,420.00,-24.11,62.0,M-LC,7.48,98.0,-0.10,1.45,24.63,XR,NTT,IT
69,TANLA,5.83,-16.93,159.04,115.17,357697.0,-45851.0,224910.0,1903.93,NaN,83.0,NaN,18.75,147.0,-0.13,1.45,98.82,AR,ATH,IT
79,VALIANTORG,3.56,-22.61,125.03,74.15,158308.0,-36989.0,126616.0,838.00,NaN,42.0,NaN,37.83,145.0,-0.23,0.82,88.42,XR,NTT,CHEMICALS
41,INDIAMART,4.92,-30.13,187.57,100.92,161635.0,-37163.0,86173.0,4675.60,NaN,56.0,NaN,16.62,140.0,-0.23,0.56,9.33,AR,ATH,MISC


In [18]:
# top 5 to accumulate based on lowest RSI_14
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'RSI_14', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
54,PAGEIND,-9.99,-14.15,54.75,32.85,126147.0,-37974.0,230405.0,50936.07,-36.12,35.0,X-MC,2.81,55.0,-0.30,1.49,10.02,X40,ATH,APPARELS
6,AWL,0.17,-42.18,188.85,67.02,389080.0,-150283.0,206026.0,485.00,-57.34,40.0,X-SC,8.70,81.0,-0.39,1.33,2.48,XY24,NTT,FMCG
37,HONAUT,0.07,-20.52,89.83,50.88,137687.0,-39563.0,153275.0,58192.79,-37.86,43.0,X-SC,7.76,90.0,-0.29,0.99,16.78,X40N,ATH,ELECTRICAL
53,NESTLEIND,-1.88,6.05,21.85,29.22,72349.0,18878.0,331116.0,1540.00,-19.91,49.0,X-LC,8.80,20.0,0.26,2.14,8.83,XY25,ATH,FMCG
33,HCLTECH,-1.43,-5.91,33.60,25.70,114122.0,-21323.0,339650.0,1837.07,-10.50,51.0,X-LC,9.03,8.0,-0.19,2.19,35.52,X40,ATH,IT


In [19]:
# top 5 to accumulate based on Spread and CumlRnk
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['CumlRnk'] < 100)].sort_values(by = 'Spread%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
17,CIPLA,-1.16,5.49,17.44,23.89,55314.0,16513.0,317168.0,1648.14,19.62,60.0,X-LC,0.67,18.0,0.30,2.05,18.80,X40N,ATH,PHARMA
1,ABBOTINDIA,2.03,-10.63,29.09,15.37,46497.0,-19015.0,159840.0,34390.26,-27.16,55.0,X-MC,0.75,58.0,-0.41,1.03,8.27,X40,ATH,PHARMA
11,BERGEPAINT,0.33,-11.91,42.26,25.32,95951.0,-30699.0,227050.0,680.00,-23.45,52.0,X-MC,2.34,68.0,-0.32,1.47,19.06,XY24,NTT,PAINTS
76,UNITDSPR,0.55,4.14,15.96,20.77,38918.0,9700.0,243844.0,1644.00,-11.27,58.0,X-MC,2.41,64.0,0.25,1.57,17.21,X40N,NTT,BREWERIES
54,PAGEIND,-9.99,-14.15,54.75,32.85,126147.0,-37974.0,230405.0,50936.07,-36.12,35.0,X-MC,2.81,55.0,-0.30,1.49,10.02,X40,ATH,APPARELS


In [20]:
# top 5 to accumulate based on lowest Gained%
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'Gained%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
6,AWL,0.17,-42.18,188.85,67.02,389080.0,-150283.0,206026.0,485.00,-57.34,40.0,X-SC,8.70,81.0,-0.39,1.33,2.48,XY24,NTT,FMCG
28,GODREJCP,-0.77,-20.72,66.62,32.10,131802.0,-51695.0,197842.0,1452.16,-22.23,57.0,X-MC,15.92,32.0,-0.39,1.28,7.19,X40N,ATH,FMCG
55,PGHH,-0.85,-46.49,144.85,31.02,165500.0,-99264.0,114256.0,17484.40,-60.69,54.0,X-SC,23.18,57.0,-0.60,0.74,7.56,X40,ATH,FMCG
20,DABUR,-0.10,-25.28,92.31,43.70,171112.0,-62701.0,185367.0,735.00,-35.48,55.0,X-MC,12.29,70.0,-0.37,1.20,7.81,XY24,BTT,FMCG
1,ABBOTINDIA,2.03,-10.63,29.09,15.37,46497.0,-19015.0,159840.0,34390.26,-27.16,55.0,X-MC,0.75,58.0,-0.41,1.03,8.27,X40,ATH,PHARMA


In [21]:
# top 5 to accumulate based on lowest CurrAlloc%
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
55,PGHH,-0.85,-46.49,144.85,31.02,165500.0,-99264.0,114256.0,17484.40,-60.69,54.0,X-SC,23.18,57.0,-0.60,0.74,7.56,X40,ATH,FMCG
2,ACC,0.06,-44.55,215.25,74.81,320034.0,-119448.0,148680.0,3906.00,-44.95,63.0,X-SC,13.83,85.0,-0.37,0.96,10.14,XY24,BTT,CEMENT
37,HONAUT,0.07,-20.52,89.83,50.88,137687.0,-39563.0,153275.0,58192.79,-37.86,43.0,X-SC,7.76,90.0,-0.29,0.99,16.78,X40N,ATH,ELECTRICAL
1,ABBOTINDIA,2.03,-10.63,29.09,15.37,46497.0,-19015.0,159840.0,34390.26,-27.16,55.0,X-MC,0.75,58.0,-0.41,1.03,8.27,X40,ATH,PHARMA
81,VOLTAS,-0.50,-12.61,66.14,45.20,110827.0,-24177.0,167565.0,1856.00,20.30,61.0,X-MC,18.97,79.0,-0.22,1.08,18.06,XY25,NTT,AC


In [22]:
# top 5 to accumulate based on CumlRnk
df_tmp = df_portfolio_features.sort_values(by = 'CumlRnk', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
46,ITC,-0.14,-18.96,58.82,28.70,208583.0,-82983.0,354612.0,452.00,-26.68,67.0,X-LC,7.18,1.0,-0.40,2.29,11.61,X40,NTT,FMCG
71,TCS,-1.74,-26.84,73.24,26.75,261534.0,-130983.0,357092.0,4208.34,-40.96,54.0,X-LC,7.79,2.0,-0.50,2.30,23.85,X40,ATH,IT
44,INFY,-0.93,-17.22,72.33,42.66,240025.0,-69008.0,331847.0,1972.00,-40.08,56.0,X-LC,7.70,3.0,-0.29,2.14,18.80,X40,NTT,IT
73,TMPV,-1.12,-26.22,102.77,49.60,361571.0,-125047.0,351825.0,600.00,-85.99,61.0,X-LC,16.38,4.0,-0.35,2.27,13.28,XY24,NTT,AUTO
33,HCLTECH,-1.43,-5.91,33.60,25.70,114122.0,-21323.0,339650.0,1837.07,-10.50,51.0,X-LC,9.03,8.0,-0.19,2.19,35.52,X40,ATH,IT


In [23]:
# top 5 for average up
df_tmp = df_portfolio_features[(df_portfolio_features['Dev%_200'] > 5)].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
75,TTKPRESTIG,-1.16,-24.37,32.35,0.09,24656.0,-24561.0,76216.0,770.00,-8.02,62.0,M-SC,1.84,250.0,-1.00,0.49,37.18,OX40N,NTT,DURABLES
12,BOSCH-HCIL,-0.50,-22.56,29.12,-0.01,28821.0,-28831.0,98974.0,2282.00,775.75,52.0,M-SC,19.74,231.0,-1.00,0.64,68.47,OX40N,BTT,AC
79,VALIANTORG,3.56,-22.61,125.03,74.15,158308.0,-36989.0,126616.0,838.00,NaN,42.0,NaN,37.83,145.0,-0.23,0.82,88.42,XR,NTT,CHEMICALS
67,SURYODAY,6.03,-9.19,35.48,23.03,57697.0,-16452.0,162619.0,216.00,-25.86,68.0,H-SC,5.82,166.0,-0.29,1.05,38.03,XR,NTT,BANKS
27,GLAXO,-2.56,6.70,27.17,35.69,57772.0,13348.0,212632.0,3380.16,-19.92,55.0,X-MC,8.18,56.0,0.23,1.37,26.37,X40,ATH,PHARMA


In [24]:
# top 5 RSP
df_tmp = df_portfolio_features.sort_values(by = 'RSP', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
69,TANLA,5.83,-16.93,159.04,115.17,357697.0,-45851.0,224910.0,1903.93,NaN,83.0,NaN,18.75,147.0,-0.13,1.45,98.82,AR,ATH,IT
83,WIPRO,-1.30,-10.20,114.93,93.01,258273.0,-25518.0,224722.0,420.00,-24.11,62.0,M-LC,7.48,98.0,-0.10,1.45,24.63,XR,NTT,IT
13,BSOFT,-0.18,-40.88,148.97,47.20,123877.0,-57497.0,83156.0,811.90,NaN,58.0,NaN,10.11,172.0,-0.46,0.54,22.72,XR,ATH,IT
50,M&M,-2.17,5.83,11.01,17.48,35082.0,17566.0,318641.0,3762.88,-7.77,74.0,X-LC,5.67,25.0,0.50,2.06,23.21,X40N,ATH,AUTO
61,REPCOHOME,-1.39,-15.96,125.87,89.82,303061.0,-45732.0,240773.0,880.00,NaN,74.0,NaN,11.59,161.0,-0.15,1.55,28.31,XY24,NTT,FINANCE


In [25]:
# Top N allocation
df_tmp = df_portfolio_features.sort_values(by = 'CurrAlloc%', ascending=False)
top_n_values = [10, 25, 50]

sum_df = pd.DataFrame({
    'Top_N': top_n_values,
    'Sum_Alloc%': [df_tmp['CurrAlloc%'].head(n).sum() for n in top_n_values]
})

sum_df

,Top_N,Sum_Alloc%
0,10,22.56
1,25,51.29
2,50,82.21


In [26]:
# market-cap-wise allocation
df_tmp = df_portfolio_features[cols]
df_tmp.groupby(df_tmp['Conviction'].str[-2:])['CurrAlloc%'].sum().sort_values(ascending=False)

Conviction
LC    42.84
SC    20.53
MC    19.65
Name: CurrAlloc%, dtype: float64

In [27]:
# criteria-wise allocation
df_portfolio_features.groupby('Criteria')['CurrAlloc%'].sum().sort_values(ascending=False)

Criteria
X40      27.07
XY24     23.03
X40N     19.19
XR       10.06
XY25      9.79
OX40N     6.77
AR        3.37
SR        0.74
Name: CurrAlloc%, dtype: float64

In [28]:
# conviction-wise allocation
df_portfolio_features.groupby('Conviction')['CurrAlloc%'].sum().sort_values(ascending=False)

Conviction
X-LC    41.39
X-MC    17.24
H-SC     8.30
X-SC     7.06
M-SC     5.17
M-LC     1.45
M-MC     1.08
H-MC     0.99
L-MC     0.34
Name: CurrAlloc%, dtype: float64

In [29]:
# sector-wise stats
df_tmp = df_portfolio_features.groupby('Category')[['CurrAlloc%', 'Current', 'Current P/L', 'FTT Amt']].sum().sort_values(by=['Current', 'Current P/L'], ascending=False)
df_tmp['Current P/L%'] = round(df_tmp['Current P/L'] * 100 / df_tmp['Current'], 2)
df_tmp['FTT%'] = round(df_tmp['FTT Amt'] * 100 / df_tmp['Current'], 2)
cols = ['CurrAlloc%', 'Current P/L%', 'FTT%']
df_tmp[cols].sort_values(by=['CurrAlloc%'], ascending=False)

,CurrAlloc%,Current P/L%,FTT%
Category,,,
FMCG,18.19,-25.04,70.14
IT,12.77,-29.60,97.12
FINANCE,11.00,-19.46,70.30
AUTO,9.29,-11.04,51.83
MISC,5.62,-42.24,99.41
PAINTS,5.55,-23.17,45.30
INSURANCE,5.47,2.59,34.02
BANKS,5.29,-9.34,64.17
PHARMA,4.45,1.57,23.14


In [30]:
# money to be made criteria-wise
df_portfolio_features.groupby('Criteria')['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

,sum,count
Criteria,,
XY24,3928569.0,19
X40,2136902.0,16
XR,1547964.0,13
X40N,1460077.0,12
AR,986692.0,6
XY25,772937.0,6
OX40N,740907.0,10
SR,298579.0,2


In [31]:
# money to be made conviction-wise
df_portfolio_features.groupby('Conviction')['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

,sum,count
Conviction,,
X-LC,2919388.0,19
X-MC,1565696.0,13
X-SC,1182592.0,6
H-SC,861874.0,8
M-SC,363066.0,6
H-MC,260586.0,1
M-LC,258273.0,1
M-MC,78507.0,1
L-MC,49805.0,1


In [32]:
# money to be made criteria and conviction-wise
df_portfolio_features.groupby(['Conviction', 'Criteria'])['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

sum  count
Conviction Criteria                  
X-LC       X40       1274995.0      8
X-SC       XY24       709114.0      2
X-LC       XY24       582027.0      2
           X40N       566067.0      5
H-SC       XY24       526915.0      3
X-MC       X40N       523795.0      4
           X40        473715.0      5
X-LC       XY25       393307.0      3
X-MC       XY25       301123.0      2
X-SC       X40        267982.0      2
X-MC       XY24       267063.0      2
H-MC       XY24       260586.0      1
M-LC       XR         258273.0      1
H-SC       OX40N      238164.0      3
X-SC       X40N       205496.0      2
M-SC       OX40N      175749.0      4
           XY24       127445.0      1
X-LC       XR         102992.0      1
H-SC       XR          96795.0      2
M-MC       XY25        78507.0      1
M-SC       XR          59872.0      1
L-MC       XR          49805.0      1

In [33]:
# notebook execution time

end_time = time.time()
execution_time = round(end_time - start_time, 0)
print(f"Notebook execution time: {execution_time} seconds")

Notebook execution time: 33.0 seconds
